In [1]:
import os
import re
import pickle
import subprocess
import numpy as np
import pandas as pd

In [4]:
with open("/home/zachkaras/fmri_recursion/midprocessing/special_char_translation.pkl", 'rb') as f:
    special_chars = pickle.load(f)

In [8]:
# onset_df = []
# with open('../data/125/relative-onsets-125-3.txt', 'r') as f:
#     for line in f:
#         newline = line.strip()
#         onset_df.append(re.split('\s', newline))
# onset_df = pd.DataFrame(onset_df)
# onset_df.columns = ['stim_id', 'timestamp']
# # print(onset_df)

# ts_to_match = float(onset_df.loc[1, 'timestamp'])
# end_ts = float(df.loc[0,'timestamp'])/10**3
# magic_number = (ts_to_match - end_ts) * (-1)
# df['timestamp'].apply(lambda x: (float(x)/10**3) - magic_number) # this gives end timestamps
# # print((ts_to_match - end_ts)*-1)


# def print_answer(timestamps, answer):
#     diff = (timestamps[-1] - timestamps[0])/10**3 # duration based on keystrokes
#     print('\nnew stimulus') 
#     print(diff) 
#     print([(float(t)/10**3) - magic_number for t in timestamps])
#     print(answer) 
#     return diff
def calculate_diff(timestamps):
    diff = (timestamps[-1] - timestamps[0])/10**3 # duration based on keystrokes
    
def calculate_stim_onset(onsetfile):
    onset_df = []
    with open(onsetfile, 'r') as f:
        for line in f:
            newline = line.strip()
            onset_df.append(re.split('\s', newline))
    onset_df = pd.DataFrame(onset_df)
    onset_df.columns = ['stim_id', 'timestamp']
    # # print(onset_df)

    ts_to_match = float(onset_df.loc[1, 'timestamp'])
    end_ts = float(onset_df.loc[0,'timestamp'])/10**3
    magic_number = (ts_to_match - end_ts) * (-1)
    # onset_df['timestamp'].apply(lambda x: (float(x)/10**3) - magic_number) # this gives end timestamps
    return magic_number
    

def process_keystrokes(keyfile, onsetfile): # return a list of tokens corresponding to keystrokes
    try:    
        with open(keyfile, 'r') as f:
            answer = '' # where the participant's response will be accumulated
            timestamps = [] # all the timestamps for each keystroke for a participant response
            adjusted_timestamps = []
            durations = [] # all the durations based on the keystroke data
            keystrokes_df = []
            magic_number = calculate_stim_onset(onsetfile) # this sets the start time of the scan to 0
            print(magic_number)
            
            lines = f.readlines()
            for i, line in enumerate(lines):
                print(line)
                asci = re.split(',', line.strip())
                keystrokes_df.append(asci)
                if (len(asci) <= 1 and i != 0) or (i == len(lines) - 1): # if it's 'new stimulus' or '<timestamp>, <ascii key>'
                    diff = calculate_diff(timestamps)              #      or the first or last stimulus
                    answer = ''
                    durations.append(diff)
                    timestamps = []
                    adjusted_timestamps = []
                elif len(asci) == 2: # if it's the comma separated timestamp and keystroke
                    asci_int = int(asci[1])
                    asci_chr = chr(asci_int)
                    ts = float(asci[0])
                    timestamps.append(ts)
                    adjusted_timestamps.append(ts - magic_number)
                    answer += asci_chr
            return pd.DataFrame(keystrokes_df)
    except:
        print("maybe no file")
    

In [9]:
# for loop to read in keystrokes
keydir = "/home/zachkaras/fmri/fmri_model/data"
keyfiles = os.listdir(keydir)
for person in keyfiles:
    
    # reading in keystroke file from long response coding
    try: # fill in the blank is category 2, long response is category 3
        keystrokes_file = f"{keydir}/{person}/keystrokes-{person}-3.txt"
        onset_file = f"{keydir}/{person}/relative-onsets-{person}-3.txt"
    except:
        print("no keystroke files")
        continue
    keystroke_df = process_keystrokes(keystrokes_file, onset_file)
    break
    



-77.9122454347741
new stimulus

272923195.327717, 73

272923755.489276, 32

272923978.149979, 67

272924228.816138, 65

272924423.129878, 66

272925466.631563, 61

272926297.549061, 8

272926469.091462, 8

272927110.235228, 66

272927902.657395, 84

272928198.652062, 32

272928799.199305, 82

272929295.071129, 69

272929652.760527, 65

272930541.750459, 70

272931350.513882, 61

272931518.944443, 68

272931859.148498, 8

272932122.178238, 8

272932275.682395, 8

272932496.755135, 83

272933186.981203, 8

272933557.792099, 68

272933724.054053, 32

272934132.319948, 73

272934291.99785, 84

272938365.05245, 16

272938933.879835, 16

272938955.743727, 16

272938990.300223, 16

272939021.087943, 16

272939052.437164, 16

272939083.731179, 16

272939115.549243, 16

272939128.930963, 17

272939161.760785, 16

272939203.609868, 16

272939224.058908, 16

272939255.254716, 16

272939286.510724, 16

272939317.8003, 16

272939348.619923, 16

272939380.520382, 16

272939426.869524, 16

272939457.

In [7]:
print(keystroke_df)

                    0     1
0        new stimulus  None
1    344965504.531269    13
2    344968903.478564    66
3    344969046.516705    79
4    344969216.505267    79
..                ...   ...
843  345533681.148826    36
844  345535260.980147    39
845  345537783.567417    16
846  345538075.420731   221
847  345538984.070438    36

[848 rows x 2 columns]
